# Data verification
Keep the official splits unchanged.

In [ ]:
from pathlib import Path
import sys
MODULE = Path(r"D:\HK261\DeepLearning\Assignment\Assignment\src\a3_image_only")
if not MODULE.is_dir(): MODULE = Path.cwd()
sys.path.insert(0, str(MODULE))
from config import Config
cfg = Config()
# cfg.split_dir = Path(r"...folder containing train.csv, validation.csv, test.csv, labels.json...")
print(cfg.splits())


In [ ]:
import torch, pandas as pd
from dataset import load_splits, make_loader
frames, labels = load_splits(cfg)
print(labels)
print({k:len(v) for k,v in frames.items()})
display(pd.DataFrame({k:v.product_type.value_counts() for k,v in frames.items()}))
print('Exact overlap checks and file existence passed')


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(3,4,figsize=(12,9))
for ax,(name,label) in zip(axes.flat,sorted(labels.items(),key=lambda z:z[1])):
    row=frames['train'].loc[frames['train'].label==label].iloc[0]
    with Image.open(row.local_path) as im: ax.imshow(im.convert('RGB'))
    ax.set_title(name,fontsize=8); ax.axis('off')
plt.tight_layout()


In [ ]:
from model import build_model
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')
x,y,i=next(iter(make_loader(cfg,frames['train'],True,device)))
print(x.shape,y.shape)
model=build_model(cfg,len(labels),pretrained=False).to(device).eval()
with torch.no_grad(): logits=model(x.to(device))
assert logits.shape==(len(y),len(labels))
print('Smoke test:',logits.shape)
del model,x,y,i,logits
if device.type=='cuda': torch.cuda.empty_cache()
